# Milestone 3 · Notebook 1: Data Preparation

**Project:** Deepfake Voice Detection for Indian Languages · Group-3 · Owner: Himanshu

This notebook builds the small subset used to check the pipeline end to end:

1. Split the speakers into train, validation and test, so that no speaker is in two of them.
2. Choose a seeded, balanced subset of clips from every data source.
3. Preprocess every clip the same way.
4. Write one manifest that lists every clip with its label and split.
5. Run the leakage checks and show the folder structure.

The output folder `data/` is uploaded as the private Kaggle dataset `tarikasubhashhedaoo/dsai-group3-m3-subset`.
`02_pipeline_check.ipynb` reads it there.

**Requirements:** `pip install -r requirements.txt`. You also need a Hugging Face login with access to the
gated `ai4bharat/Kathbath`, and about 4 GB of free disk. The run takes about 30–40 minutes, mostly
downloading.

## 1. Setup

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless tqdm notice

import hashlib
import io
import json
import shutil
import subprocess
import tarfile
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from math import gcd
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from huggingface_hub import HfFileSystem, hf_hub_download
from scipy.signal import resample_poly

pd.set_option("display.width", 200)
fs = HfFileSystem()

All sources are pinned to a fixed revision, so a re-run reads the same files.

In [2]:
SEED = 42
LANGUAGES = {"hi": "Hindi", "ta": "Tamil", "te": "Telugu"}

INDICSYNTH = "datasets/vdivyasharma/IndicSynth@c0a10386b723717aff682f757bd67f72983f269f"
KATHBATH = "datasets/ai4bharat/Kathbath@5b9e92849222026d9141acba4e8434fe816396bf"
GENERATED = "datasets/TarikaHedaoo/dsai-group3-fake-speech@b9db125b305701982c76b20ef613217859f43c42"
ASVSPOOF = "datasets/SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA@401e90d3efddb3a2b2ee5f3ee53b737b603d7eae"
IN_THE_WILD = "datasets/mueller91/In-The-Wild@eee168f92c367f8c82ff2cf42b6f61e362fd6211"
ASVSPOOF_FULL = "datasets/Bisher/ASVspoof_2019_LA@aea92dd83a9c56e070c0b1e9f02e7c0d96216a4c"  # has train and dev
KATHBATH_TEST_TAR = "https://objectstore.e2enetworks.net/indic-superb/kathbath/clean/testunk_audio.tar"

# Subset size, in clips
PER_LANGUAGE = {"train": 200, "valid": 50, "test_indomain": 50}  # real; fakes use the same numbers
UNSEEN_PER_GENERATOR = 30   # sentences per generator and language
ITW_PER_LABEL = 100
ASVSPOOF_PER_LABEL = {"test_asvspoof": 50, "train": 150, "valid": 50}
CLIPS_PER_ROW_GROUP = 10    # spread the picks over many parquet row groups (and so many speakers)

REPO = Path.cwd().resolve().parents[1]
DATA = REPO / "data"
RAW, PROCESSED, MANIFESTS = DATA / "raw", DATA / "processed", DATA / "manifests"
for folder in (RAW, PROCESSED, MANIFESTS):
    folder.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)
print("Data folder:", DATA)

Data folder: /home/himanshu/myproject/dsai-project/data


## 2. Preprocessing

This is the same as the Milestone-2 `preprocess.py`. Every clip, real or fake, goes through these steps,
so sample rate, format, edge silence and loudness can't give its label away:

1. Decode and mix to mono. Kathbath's test recordings are m4a, so `ffmpeg` decodes them.
2. Resample to 16 kHz, the rate XLS-R was trained on.
3. Trim edge silence: 20 ms frames that are more than 40 dB below the loudest frame.
4. Drop the clip if less than 1 s is left.
5. Set the loudness to −20 dBFS RMS, limiting the peak to 0.99.
6. Store as 16-bit PCM WAV and hash the samples (SHA-1) to find exact duplicates.

In [3]:
TARGET_SR, TARGET_DBFS, SILENCE_DB, FRAME_S, MIN_SECONDS = 16000, -20.0, 40, 0.02, 1.0


def decode(data):
    try:
        audio, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=True)
    except sf.LibsndfileError:  # m4a
        # ffmpeg reads from a file, not a pipe: some m4a files keep their index at the end, and from a pipe
        # those decode to nothing
        ffmpeg = shutil.which("ffmpeg") or __import__("imageio_ffmpeg").get_ffmpeg_exe()
        with tempfile.NamedTemporaryFile(suffix=".m4a") as tmp:
            tmp.write(data)
            tmp.flush()
            wav = subprocess.run([ffmpeg, "-v", "error", "-i", tmp.name, "-f", "wav", "pipe:1"],
                                 capture_output=True, check=True).stdout
        audio, sr = sf.read(io.BytesIO(wav), dtype="float32", always_2d=True)
    return audio.mean(axis=1), sr


def trim_silence(audio):
    frame = int(FRAME_S * TARGET_SR)
    n = len(audio) // frame
    if n == 0:
        return audio
    energy_db = 10 * np.log10((audio[:n * frame].reshape(n, frame) ** 2).mean(axis=1) + 1e-12)
    voiced = np.where(energy_db > energy_db.max() - SILENCE_DB)[0]
    return audio[voiced[0] * frame:(voiced[-1] + 1) * frame]


def normalise(audio):
    rms = np.sqrt(np.mean(audio ** 2))
    audio = audio * (10 ** (TARGET_DBFS / 20) / rms)
    peak = np.abs(audio).max()
    return audio / peak * 0.99 if peak > 0.99 else audio


def preprocess(data):
    """Raw file bytes -> (int16 samples at 16 kHz or None if dropped, original sample rate)."""
    audio, sr = decode(data)
    if sr != TARGET_SR:
        g = gcd(sr, TARGET_SR)
        audio = resample_poly(audio, TARGET_SR // g, sr // g).astype(np.float32)
    audio = trim_silence(audio)
    if len(audio) < MIN_SECONDS * TARGET_SR:
        return None, sr
    return np.round(normalise(audio) * 32767).astype(np.int16), sr

`save()` keeps each clip twice, the original under `raw/` and the preprocessed WAV under `processed/`,
and adds one row to the manifest. The manifest columns follow the format fixed in Milestone 2
(Section 8), plus `raw_path` and `text`.

In [4]:
rows, dropped = [], []


def save(clip, data, raw_name):
    """clip: dict with clip_id, dataset, language, label, generator, speaker_id, ref_speaker_id, gender, text, split."""
    raw = RAW / clip["dataset"] / clip["language"] / raw_name
    raw.parent.mkdir(parents=True, exist_ok=True)
    raw.write_bytes(data)
    pcm, sr = preprocess(data)
    if pcm is None:
        dropped.append({**clip, "reason": "shorter than 1 s after trimming"})
        return
    out = PROCESSED / clip["dataset"] / clip["language"] / f"{clip['clip_id']}.wav"
    out.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out, pcm, TARGET_SR, subtype="PCM_16")
    rows.append({**clip, "path": str(out.relative_to(DATA)), "raw_path": str(raw.relative_to(DATA)),
                 "duration_s": round(len(pcm) / TARGET_SR, 3), "orig_sr": sr,
                 "sha1": hashlib.sha1(pcm.tobytes()).hexdigest()})

## 3. Speaker-Disjoint Split

IndicSynth was built only from Kathbath's **train** speakers. Its fakes imitate those speakers, using
their own recordings as references. So the real and fake clips of one speaker must stay in the same split.

The split is the one simulated in the Milestone-2 IndicSynth notebook (Section 5), with the same code and
seed. It puts 70% of each language's speakers in train, 15% in validation and 15% in test, separately for
female and male speakers. The speakers come from the IndicSynth metadata. Reading it from Hugging Face
takes about an hour, so the copy saved by the Milestone-2 notebook is used
(`data/raw/indicsynth/indicsynth_metadata_raw.parquet`).

In [5]:
meta = pd.read_parquet(RAW / "indicsynth" / "indicsynth_metadata_raw.parquet")
meta["source_gender"] = meta.source_ref.str.split("-").str[2].str[0]  # '..-1087-f.wav' -> 'f'

split_rng = np.random.default_rng(SEED)
speaker_rows = []
for language in LANGUAGES.values():
    part = meta[meta.language == language]
    genders = part.groupby("target_speaker").gender.first()
    for source in part.source_speaker.dropna().unique():  # speakers that only appear as a FreeVC24 source
        if source not in genders.index:
            genders[source] = "Female" if part[part.source_speaker == source].source_gender.iloc[0] == "f" else "Male"
    for gender in ["Female", "Male"]:
        ids = np.sort(genders[genders == gender].index.to_numpy())
        split_rng.shuffle(ids)
        n_train, n_valid = round(0.70 * len(ids)), round(0.15 * len(ids))
        for i, speaker in enumerate(ids):
            split = "train" if i < n_train else ("valid" if i < n_train + n_valid else "test_indomain")
            speaker_rows.append({"language": language, "speaker_id": int(speaker), "gender": gender, "split": split})

speakers = pd.DataFrame(speaker_rows)
speakers.to_csv(MANIFESTS / "speaker_split.csv", index=False)
split_of = {(s.language, s.speaker_id): s.split for s in speakers.itertuples()}
pd.crosstab([speakers.language, speakers.gender], speakers.split)[["train", "valid", "test_indomain"]]

split            train  valid  test_indomain
language gender                             
Hindi    Female     37      8              8
         Male       34      7              7
Tamil    Female     22      5              5
         Male       74     16             16
Telugu   Female     29      6              6
         Male       30      6              7

These match Table 5.1 of the Milestone-2 notebook. For example, Hindi has 71 / 15 / 15 speakers.

## 4. Choosing and Preparing the Clips

| Split | Real | Fake |
| --- | --- | --- |
| `train` / `valid` / `test_indomain` | Kathbath, 200 / 50 / 50 per language | IndicSynth, the same numbers per language, half XTTS-v2 and half FreeVC24 |
| `test_unseen_gen` | Kathbath **test** speakers reading the same sentences, 90 per language | Our generated set, 30 sentences per generator and language |
| `test_itw` | In-the-Wild, 100 | In-the-Wild, 100 |
| `test_asvspoof` | ASVspoof 2019 LA eval, 50 | ASVspoof 2019 LA eval, 50 |
| `train` / `valid` (English) | ASVspoof 2019 LA train / dev, 150 / 50 | ASVspoof 2019 LA train / dev, 150 / 50 |

IndicSynth and Kathbath are large parquet files. Only their small text columns are read in full.
The picks come from row groups chosen at random, at most 10 clips from each, and only those row groups
have their audio downloaded.

In [6]:
def pick(candidates, n):
    """n random rows, at most CLIPS_PER_ROW_GROUP from each randomly chosen (file, row_group)."""
    groups = list(candidates.groupby(["file", "row_group"]).groups.values())
    chosen, total = [], 0
    for g in rng.permutation(len(groups)):
        take = candidates.loc[groups[g]].sample(min(CLIPS_PER_ROW_GROUP, n - total, len(groups[g])),
                                                random_state=int(rng.integers(2**31)))
        chosen.append(take)
        total += len(take)
        if total == n:
            break
    return pd.concat(chosen)


def read_columns(path, columns):
    """The given (small) columns of one parquet file, with each row's row group and position in it."""
    with fs.open(path, cache_type="none") as f:
        parquet = pq.ParquetFile(f, pre_buffer=True)
        part = parquet.read(columns=columns).to_pandas()
        sizes = [parquet.metadata.row_group(i).num_rows for i in range(parquet.metadata.num_row_groups)]
    part["file"] = path
    part["row_group"] = np.repeat(np.arange(len(sizes)), sizes)
    part["row_in_group"] = np.concatenate([np.arange(n) for n in sizes])
    return part


def read_audio(picks, column):
    """Yield (row, audio bytes), downloading each needed row group once (6 in parallel)."""
    def load(key):
        with fs.open(key[0], cache_type="none") as f:
            table = pq.ParquetFile(f, pre_buffer=True).read_row_group(key[1], columns=[column])
        return key, table.column(column).to_pylist()

    keys = list(picks.groupby(["file", "row_group"]).groups)
    with ThreadPoolExecutor(6) as pool:
        for (file, group), audio in pool.map(load, keys):
            for row in picks[(picks.file == file) & (picks.row_group == group)].itertuples():
                yield row, audio[row.row_in_group]["bytes"]

### 4.1 IndicSynth (Fake, Hindi/Tamil/Telugu)

A clip is used only if all its speakers are in one split: the target speaker, and for FreeVC24 also
the source speaker. That's the Milestone-2 rule.

In [7]:
t0 = time.time()
meta["split"] = [split_of[(l, s)] for l, s in zip(meta.language, meta.target_speaker)]
source_split = [split_of[(l, s)] if pd.notna(s) else None for l, s in zip(meta.language, meta.source_speaker)]
usable = meta[[src is None or src == tgt for src, tgt in zip(source_split, meta.split)]].copy()
usable["file"] = INDICSYNTH + "/" + usable.language + "/" + usable.file

fake_picks = pd.concat([
    pick(usable[(usable.language == language) & (usable.generator == generator) & (usable.split == split)], n // 2)
    .assign(lang=code)
    for code, language in LANGUAGES.items() for split, n in PER_LANGUAGE.items() for generator in ["xtts_v2", "freevc24"]
])
for r, data in read_audio(fake_picks, "audio"):
    save({"clip_id": f"indicsynth_{r.lang}_{Path(r.audio_path).stem}", "dataset": "indicsynth", "language": r.lang,
          "label": "fake", "generator": r.generator, "speaker_id": r.target_speaker,
          "ref_speaker_id": r.source_speaker if pd.notna(r.source_speaker) else r.target_speaker,
          "gender": r.gender[0].lower(), "text": r.transcript if isinstance(r.transcript, str) else "",
          "split": r.split}, data, r.audio_path)
print(f"IndicSynth: {len(fake_picks)} clips chosen, {time.time() - t0:.0f} s")

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/vdivyasharma/IndicSynth/resolve/c0a10386b723717aff682f757bd67f72983f269f/Telugu/train-00066-of-00129.parquet


Retrying in 1s [Retry 1/5].


IndicSynth: 900 clips chosen, 109 s


### 4.2 Kathbath (Real, Train Speakers)

Only the small text columns of every parquet file are read first, to find each recording's speaker.
Each recording then gets its speaker's split.

In [8]:
t0 = time.time()
paths = [p for folder in ["hindi", "tamil", "telugu"] for p in sorted(fs.glob(f"{KATHBATH}/{folder}/train-*.parquet"))]
with ThreadPoolExecutor(8) as pool:  # about 20 s per file, so 8 files are read at a time
    columns = ["fname", "text", "lang", "gender", "speaker_id"]
    kathbath = pd.concat(pool.map(lambda path: read_columns(path, columns), paths), ignore_index=True)
kathbath["split"] = [split_of.get((LANGUAGES[l], s)) for l, s in zip(kathbath.lang, kathbath.speaker_id)]
print(f"Kathbath train recordings: {len(kathbath):,}; recordings of speakers not in IndicSynth (unused): "
      f"{kathbath.split.isna().sum():,}")

real_picks = pd.concat([pick(kathbath[(kathbath.lang == code) & (kathbath.split == split)], n)
                        for code in LANGUAGES for split, n in PER_LANGUAGE.items()])
for r, data in read_audio(real_picks, "audio_filepath"):
    stem = Path(r.fname).stem
    save({"clip_id": f"kathbath_{r.lang}_{stem}", "dataset": "kathbath", "language": r.lang, "label": "real",
          "generator": "none", "speaker_id": r.speaker_id, "ref_speaker_id": r.speaker_id,
          "gender": r.gender[0].lower(), "text": r.text, "split": r.split}, data, f"{stem}.flac")
print(f"Kathbath: {len(real_picks)} clips chosen, {time.time() - t0:.0f} s")

Kathbath train recordings: 258,187; recordings of speakers not in IndicSynth (unused): 0


Kathbath: 900 clips chosen, 644 s


### 4.3 Unseen-Generator Test: Our Generated Set + Kathbath Test Speakers

Our generated set speaks sentences from Kathbath's **test** split, whose speakers are not in IndicSynth or
in training. For each language, 90 sentences are chosen, 30 per generator. The real side is the Kathbath
recording of the same sentence, so both classes say the same words. The Hugging Face copy of Kathbath
has no test split, so those recordings come from Kathbath's own archive (2.2 GB).

In [9]:
t0 = time.time()
lines = []
for path in fs.glob(f"{GENERATED}/manifests/*/*/*.jsonl"):
    lines += [json.loads(l) for l in fs.open(path).read().decode().splitlines() if l.strip()]
generated = pd.DataFrame(lines).drop_duplicates("clip_id")
generated = generated[~generated.flag_silent]

unseen = []
for code in LANGUAGES:
    sentences = rng.permutation(sorted(generated[generated.lang == code].sent_id.unique()))
    for i, generator in enumerate(["indic-parler-tts", "indicf5", "mms-tts"]):
        chosen = sentences[i * UNSEEN_PER_GENERATOR:(i + 1) * UNSEEN_PER_GENERATOR]
        unseen.append(generated[(generated.lang == code) & (generated.generator == generator)
                                & generated.sent_id.isin(chosen)])
unseen = pd.concat(unseen)

for r in unseen.itertuples():
    save({"clip_id": r.clip_id, "dataset": "generated", "language": r.lang, "label": "fake", "generator": r.generator,
          "speaker_id": r.voice_id, "ref_speaker_id": r.voice_id, "gender": r.voice_gender, "text": r.text,
          "split": "test_unseen_gen"}, fs.open(f"{GENERATED}/{r.path}").read(), Path(r.path).name)

tar_path = RAW / "kathbath" / "testunk_audio.tar"
if not tar_path.exists():
    subprocess.run(["curl", "-s", "-o", str(tar_path), KATHBATH_TEST_TAR], check=True)
wanted = {f"kb_data_clean_m4a/{LANGUAGES[r.lang].lower()}/test/audio/{r.kb_file}": r for r in unseen.itertuples()}
with tarfile.open(tar_path) as tar:
    for member in tar:
        if member.name in wanted:
            r = wanted[member.name]
            stem = Path(r.kb_file).stem
            speaker, gender = stem.split("-")[1:]
            save({"clip_id": f"kathbath_{r.lang}_{stem}", "dataset": "kathbath", "language": r.lang, "label": "real",
                  "generator": "none", "speaker_id": int(speaker), "ref_speaker_id": int(speaker), "gender": gender,
                  "text": r.text, "split": "test_unseen_gen"}, tar.extractfile(member).read(), r.kb_file)
print(f"Unseen-generator test: {len(unseen)} fake + {len(wanted)} real chosen, {time.time() - t0:.0f} s")

Unseen-generator test: 270 fake + 270 real chosen, 241 s


### 4.4 In-the-Wild (English, Test Only)

In [10]:
t0 = time.time()
with zipfile.ZipFile(fs.open(f"{IN_THE_WILD}/release_in_the_wild.zip")) as zf:
    itw = pd.read_csv(zf.open("release_in_the_wild/meta.csv"))
itw["label"] = itw.label.map({"bona-fide": "real", "spoof": "fake"})
itw = pd.concat([itw[itw.label == label].sample(ITW_PER_LABEL, random_state=SEED) for label in ["real", "fake"]])


def read_itw(name):  # one ranged read per file, so several run in parallel
    with zipfile.ZipFile(fs.open(f"{IN_THE_WILD}/release_in_the_wild.zip")) as zf:
        return zf.read(f"release_in_the_wild/{name}")


with ThreadPoolExecutor(8) as pool:
    for r, data in zip(itw.itertuples(), pool.map(read_itw, itw.file)):
        save({"clip_id": f"itw_{Path(r.file).stem}", "dataset": "in_the_wild", "language": "en", "label": r.label,
              "generator": "unknown_tts" if r.label == "fake" else "none", "speaker_id": r.speaker,
              "ref_speaker_id": r.speaker, "gender": "", "text": "", "split": "test_itw"}, data, r.file)
print(f"In-the-Wild: {len(itw)} clips chosen, {time.time() - t0:.0f} s")

In-the-Wild: 200 clips chosen, 162 s


### 4.5 ASVspoof 2019 LA (English)

**Eval (`test_asvspoof`)** comes from the copy explored in Milestone 2
(`SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA`), which has only the eval partition. Its parquet row groups are
read in random order until 50 bona fide and 50 spoof clips are found. Labels and attacks come from the official
eval protocol.

**Train and dev** answer the reviewer's Milestone-2 point. They come from `Bisher/ASVspoof_2019_LA`, a full copy
of the corpus that Vikram found and checked. Its counts match the official protocol: train 25,380 clips
(2,580 bona fide) and dev 24,844 clips (2,548 bona fide), with no speaker shared between partitions. Dev is
our `valid`. ASVspoof's own partitions are kept, so the English speakers are disjoint by design.

In [11]:
t0 = time.time()
protocol_cols = ["speaker", "utt", "unused", "attack", "key"]
protocol = pd.read_csv(hf_hub_download("SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA",
                                       "protocols/ASVspoof2019.LA.cm.eval.trl.txt", repo_type="dataset",
                                       revision=ASVSPOOF.split("@")[1]), sep=" ", names=protocol_cols).set_index("utt")


def save_asvspoof(utt, data, split):
    r = protocol.loc[utt]
    label = "real" if r.key == "bonafide" else "fake"
    save({"clip_id": f"asvspoof_{utt}", "dataset": "asvspoof2019_la", "language": "en", "label": label,
          "generator": r.attack if label == "fake" else "none", "speaker_id": r.speaker, "ref_speaker_id": r.speaker,
          "gender": "", "text": "", "split": split}, data, f"{utt}.flac")
    return label


groups = [(path, g) for path in sorted(fs.glob(f"{ASVSPOOF}/data/test-*.parquet"))
          for g in range(pq.ParquetFile(fs.open(path)).metadata.num_row_groups)]
need = {"real": ASVSPOOF_PER_LABEL["test_asvspoof"], "fake": ASVSPOOF_PER_LABEL["test_asvspoof"]}
for i in rng.permutation(len(groups)):
    path, g = groups[i]
    with fs.open(path, cache_type="none") as f:
        table = pq.ParquetFile(f, pre_buffer=True).read_row_group(g, columns=["path", "audio"]).to_pylist()
    for j in rng.permutation(len(table)):
        item = table[j]
        utt = Path(item["path"]).stem
        label = "real" if protocol.loc[utt, "key"] == "bonafide" else "fake"
        if need[label] > 0:
            need[label] -= 1
            save_asvspoof(utt, item["audio"]["bytes"], "test_asvspoof")
    if not any(need.values()):
        break
print(f"ASVspoof eval: done, {time.time() - t0:.0f} s")

t0 = time.time()
for part, split in [("train", "train"), ("validation", "valid")]:
    path = f"{ASVSPOOF_FULL}/data/{part}-00000-of-00001.parquet"
    parquet = pq.ParquetFile(fs.open(path))
    need = {0: ASVSPOOF_PER_LABEL[split], 1: ASVSPOOF_PER_LABEL[split]}  # key: 0 = bonafide, 1 = spoof
    for g in rng.permutation(parquet.metadata.num_row_groups):  # random blocks of 100 clips
        block = parquet.read_row_group(int(g)).to_pylist()
        for key in (0, 1):
            for item in [b for b in block if b["key"] == key][:min(CLIPS_PER_ROW_GROUP, need[key])]:
                need[key] -= 1
                label = "real" if key == 0 else "fake"
                save({"clip_id": f"asvspoof_{item['audio_file_name']}", "dataset": "asvspoof2019_la", "language": "en",
                      "label": label, "generator": item["system_id"] if label == "fake" else "none",
                      "speaker_id": item["speaker_id"], "ref_speaker_id": item["speaker_id"], "gender": "", "text": "",
                      "split": split}, item["audio"]["bytes"], f"{item['audio_file_name']}.flac")
        if not any(need.values()):
            break
print(f"ASVspoof train and dev: done, {time.time() - t0:.0f} s")

ASVspoof eval: done, 19 s


ASVspoof train and dev: done, 1279 s


## 5. The Manifest

In [12]:
COLUMNS = ["clip_id", "path", "raw_path", "dataset", "language", "label", "generator", "speaker_id",
           "ref_speaker_id", "gender", "text", "duration_s", "orig_sr", "sha1", "split"]
SPLITS = ["train", "valid", "test_indomain", "test_unseen_gen", "test_itw", "test_asvspoof"]
manifest = pd.DataFrame(rows)[COLUMNS]
manifest["split"] = pd.Categorical(manifest.split, SPLITS)
manifest = manifest.sort_values(["split", "dataset", "language", "clip_id"])
manifest.to_csv(MANIFESTS / "manifest_subset.csv", index=False)
pd.DataFrame(dropped).to_csv(MANIFESTS / "dropped.csv", index=False)
print(f"{len(manifest)} clips kept, {len(dropped)} dropped, {manifest.duration_s.sum() / 3600:.2f} h after preprocessing")
manifest.head(3)

3018 clips kept, 22 dropped, 4.73 h after preprocessing


,clip_id,path,raw_path,dataset,language,label,generator,speaker_id,ref_speaker_id,gender,text,duration_s,orig_sr,sha1,split
2656,asvspoof_LA_T_1004644,processed/asvspoof2019_la/en/asvspoof_LA_T_100...,raw/asvspoof2019_la/en/LA_T_1004644.flac,asvspoof2019_la,en,fake,A01,LA_0079,LA_0079,,,1.64,16000,0470d4d2ed6c7d3651d40569adce353e3047793b,train
2765,asvspoof_LA_T_1026006,processed/asvspoof2019_la/en/asvspoof_LA_T_102...,raw/asvspoof2019_la/en/LA_T_1026006.flac,asvspoof2019_la,en,fake,A01,LA_0086,LA_0086,,,3.88,16000,9a02b90ef9659b9582091a73af62e0ce037a7f39,train
2766,asvspoof_LA_T_1035222,processed/asvspoof2019_la/en/asvspoof_LA_T_103...,raw/asvspoof2019_la/en/LA_T_1035222.flac,asvspoof2019_la,en,fake,A01,LA_0086,LA_0086,,,3.46,16000,e6d29e94f8f068a4672139cefc9d0b7f631651a1,train


In [13]:
pd.crosstab([manifest.split, manifest.dataset], manifest.label, margins=True, margins_name="total")

label                            fake  real  total
split           dataset                           
train           asvspoof2019_la   146   146    292
                indicsynth        600     0    600
                kathbath            0   600    600
valid           asvspoof2019_la    50    47     97
                indicsynth        150     0    150
                kathbath            0   150    150
test_indomain   indicsynth        150     0    150
                kathbath            0   150    150
test_unseen_gen generated         270     0    270
                kathbath            0   270    270
test_itw        in_the_wild        96    97    193
test_asvspoof   asvspoof2019_la    48    48     96
total                            1510  1508   3018

In [14]:
pd.crosstab([manifest.split, manifest.language], manifest.generator)

generator                 A01  A02  A03  A04  A05  A06  A07  A08  A09  A10  ...  A17  A18  A19  freevc24  indic-parler-tts  indicf5  mms-tts  none  unknown_tts  xtts_v2
split           language                                                    ...                                                                                         
train           en         39   10   19   30   48    0    0    0    0    0  ...    0    0    0         0                 0        0        0   146            0        0
                hi          0    0    0    0    0    0    0    0    0    0  ...    0    0    0       100                 0        0        0   200            0      100
                ta          0    0    0    0    0    0    0    0    0    0  ...    0    0    0       100                 0        0        0   200            0      100
                te          0    0    0    0    0    0    0    0    0    0  ...    0    0    0       100                 0        0        0   200            0      100
valid           en          0    0    0   30    0   20    0    0    0    0  ...    0    0    0         0                 0        0        0    47            0        0
                hi          0    0    0    0    0    0    0    0    0    0  ...    0    0    0        25                 0        0        0    50            0       25
                ta          0    0    0    0    0    0    0    0    0    0  ...    0    0    0        25                 0        0        0    50            0       25
                te          0    0    0    0    0    0    0    0    0    0  ...    0    0    0        25                 0        0        0    50            0       25
test_indomain   hi          0    0    0    0    0    0    0    0    0    0  ...    0    0    0        25                 0        0        0    50            0       25
                ta          0    0    0    0    0    0    0    0    0    0  ...    0    0    0        25                 0        0        0    50            0       25
                te          0    0    0    0    0    0    0    0    0    0  ...    0    0    0        25                 0        0        0    50            0       25
test_unseen_gen hi          0    0    0    0    0    0    0    0    0    0  ...    0    0    0         0                30       30       30    90            0        0
                ta          0    0    0    0    0    0    0    0    0    0  ...    0    0    0         0                30       30       30    90            0        0
                te          0    0    0    0    0    0    0    0    0    0  ...    0    0    0         0                30       30       30    90            0        0
test_itw        en          0    0    0    0    0    0    0    0    0    0  ...    0    0    0         0                 0        0        0    97           96        0
test_asvspoof   en          0    0    0    0    0    0    4    5    2    3  ...    1    4    4         0                 0        0        0    48            0        0

[16 rows x 26 columns]

Clips dropped by preprocessing (less than 1 s left after trimming silence):

In [15]:
pd.DataFrame(dropped, columns=["clip_id", "dataset", "split", "reason"])

,clip_id,dataset,split,reason
0,itw_7055,in_the_wild,test_itw,shorter than 1 s after trimming
1,itw_6872,in_the_wild,test_itw,shorter than 1 s after trimming
2,itw_27183,in_the_wild,test_itw,shorter than 1 s after trimming
3,itw_30698,in_the_wild,test_itw,shorter than 1 s after trimming
4,itw_3819,in_the_wild,test_itw,shorter than 1 s after trimming
5,itw_3845,in_the_wild,test_itw,shorter than 1 s after trimming
6,itw_1748,in_the_wild,test_itw,shorter than 1 s after trimming
7,asvspoof_LA_E_1617147,asvspoof2019_la,test_asvspoof,shorter than 1 s after trimming
8,asvspoof_LA_E_3996568,asvspoof2019_la,test_asvspoof,shorter than 1 s after trimming
9,asvspoof_LA_E_3197194,asvspoof2019_la,test_asvspoof,shorter than 1 s after trimming


Every original sample rate is now 16 kHz, with one format and one loudness:

In [16]:
pd.crosstab(manifest.dataset, manifest.orig_sr)

orig_sr,16000,24000,44100
dataset,,,
asvspoof2019_la,485,0,0
generated,90,90,90
in_the_wild,193,0,0
indicsynth,0,900,0
kathbath,900,0,270


## 6. Leakage Checks

| # | Check | Why |
| --- | --- | --- |
| 1 | No Indian-language speaker in two of `train`, `valid`, `test_indomain` (the FreeVC24 source speaker counts too) | The model must not be tested on voices it trained on |
| 2 | No speaker of `test_unseen_gen` in `train` | That test is meant to be completely new speakers and generators |
| 3 | No exact duplicate clips (same SHA-1) | The same audio must not be in two places |
| 4 | No ASVspoof speaker in two partitions | The official protocol keeps them apart |
| 5 | `test_unseen_gen` sentences that also appear in `train` | Reported, not required to be 0: the same words read by different people |

In [17]:
def speaker_set(split):
    part = manifest[(manifest.split == split) & manifest.language.isin(LANGUAGES)]
    return set(part.speaker_id.astype(str)) | set(part.ref_speaker_id.astype(str))


asv = manifest[manifest.dataset == "asvspoof2019_la"]
asv_speakers = [set(asv[asv.split == s].speaker_id) for s in asv.split.unique()]
train_text = set(manifest[(manifest.split == "train") & (manifest.text != "")].text)

leakage = pd.DataFrame([
    ("speakers in both train and valid", len(speaker_set("train") & speaker_set("valid"))),
    ("speakers in both train and test_indomain", len(speaker_set("train") & speaker_set("test_indomain"))),
    ("speakers in both valid and test_indomain", len(speaker_set("valid") & speaker_set("test_indomain"))),
    ("speakers in both train and test_unseen_gen", len(speaker_set("train") & speaker_set("test_unseen_gen"))),
    ("duplicate clips (same SHA-1)", int(manifest.sha1.duplicated().sum())),
    ("ASVspoof speakers in two partitions",
     sum(len(a & b) for i, a in enumerate(asv_speakers) for b in asv_speakers[i + 1:])),
    ("test_unseen_gen sentences also in train (reported only)",
     len(set(manifest[manifest.split == "test_unseen_gen"].text) & train_text)),
], columns=["check", "count"])
leakage.to_csv(MANIFESTS / "leakage_report.csv", index=False)
leakage

,check,count
0,speakers in both train and valid,0
1,speakers in both train and test_indomain,0
2,speakers in both valid and test_indomain,0
3,speakers in both train and test_unseen_gen,0
4,duplicate clips (same SHA-1),0
5,ASVspoof speakers in two partitions,0
6,test_unseen_gen sentences also in train (repor...,0


## 7. Folder Structure

In [18]:
def tree(folder, depth, prefix=""):
    items = sorted(p for p in folder.iterdir() if not p.name.startswith("."))
    for i, p in enumerate(items):
        branch, extend = ("└── ", "    ") if i == len(items) - 1 else ("├── ", "│   ")
        if p.is_dir():
            print(f"{prefix}{branch}{p.name}/  ({sum(1 for f in p.rglob('*') if f.is_file())} files)")
            if depth > 1:
                tree(p, depth - 1, prefix + extend)
        else:
            print(f"{prefix}{branch}{p.name}")


print("data/")
tree(DATA, 3)

data/
├── README.md
├── manifests/  (4 files)
│   ├── dropped.csv
│   ├── leakage_report.csv
│   ├── manifest_subset.csv
│   └── speaker_split.csv
├── processed/  (3018 files)
│   ├── asvspoof2019_la/  (485 files)
│   │   └── en/  (485 files)
│   ├── generated/  (270 files)
│   │   ├── hi/  (90 files)
│   │   ├── ta/  (90 files)
│   │   └── te/  (90 files)
│   ├── in_the_wild/  (193 files)
│   │   └── en/  (193 files)
│   ├── indicsynth/  (900 files)
│   │   ├── hi/  (300 files)
│   │   ├── ta/  (300 files)
│   │   └── te/  (300 files)
│   └── kathbath/  (1170 files)
│       ├── hi/  (390 files)
│       ├── ta/  (390 files)
│       └── te/  (390 files)
└── raw/  (3042 files)
    ├── asvspoof2019_la/  (500 files)
    │   └── en/  (500 files)
    ├── generated/  (270 files)
    │   ├── hi/  (90 files)
    │   ├── ta/  (90 files)
    │   └── te/  (90 files)
    ├── in_the_wild/  (200 files)
    │   └── en/  (200 files)
    ├── indicsynth/  (901 files)
    │   ├── hi/  (300 files)
    │   

## 8. Summary

The subset, the manifest and the leakage report are in `data/`. Section 5 shows the numbers and Section 6
the checks. Next, `data/` is uploaded as the Kaggle dataset `dsai-group3-m3-subset` for
`02_pipeline_check.ipynb`.